In [16]:
import numpy as np

from scipy.optimize import nnls
from scipy.optimize import lsq_linear
from scipy.optimize import Bounds, LinearConstraint, linprog, minimize

#import from src_code.main import *
#from main import *


A = np.array([
    [1.0, 0.0],
    [1.0, 1.0],
    [0.0, 1.0]
])

b = np.array([1.0, 2.0, 1.0])

x, residual_norm = nnls(A, b)


print("x =", x)
print("||Ax-b|| =", residual_norm)
print("Ax =", A @ x)

x, residual_norm = lsq_linear(A, b).x, np.linalg.norm(A @ lsq_linear(A, b).x - b)


print("x =", x)
print("||Ax-b|| =", residual_norm)
print("Ax =", A @ x)

x = [1. 1.]
||Ax-b|| = 0.0
Ax = [1. 2. 1.]
x = [1. 1.]
||Ax-b|| = 3.1401849173675503e-16
Ax = [1. 2. 1.]


Now we want to study the transport equation as a variational problem. We can write the probabilities as a matrix, and solve for $\Gamma_{mn}$. 

In [5]:
def construct_transport_matrix(p, allowed_edges=None):
    """
    Construct A(p) such that

        dp_dt = A(p) @ gamma

    with Gamma[m, n] denoting the rate n -> m.

    Parameters
    ----------
    p : array_like, shape (N,)
        Current probabilities.

    allowed_edges : iterable of (m, n), optional
        Allowed directed transitions n -> m.
        If None, all transitions with m != n are included.

    Returns
    -------
    A : ndarray, shape (N, number_of_edges)
        Transport matrix.

    edges : list of tuples
        edges[j] = (m, n), meaning gamma[j] = Gamma[m, n].
    """
    p = np.asarray(p, dtype=float)
    N = len(p)

    if allowed_edges is None:
        edges = [
            (m, n)
            for m in range(N)
            for n in range(N)
            if m != n
        ]
    else:
        edges = list(allowed_edges)

    A = np.zeros((N, len(edges)))

    for j, (m, n) in enumerate(edges):
        if m == n:
            raise ValueError("Self-transitions Gamma[m,m] are not included.")

        # Gain in state m from n -> m
        A[m, j] = p[n]

        # Loss from state n due to n -> m
        A[n, j] = -p[n]

    return A, edges


def vector_to_rate_matrix(gamma, edges, N):
    """
    Convert the rate vector into a matrix Gamma, where
    Gamma[m,n] is the rate n -> m.
    """
    Gamma = np.zeros((N, N))

    for value, (m, n) in zip(gamma, edges):
        Gamma[m, n] = value

    return Gamma


def solve_nonnegative_rates(p, dp_dt, allowed_edges=None):
    """
    Find nonnegative rates satisfying, as closely as possible,

        dp_dt = A(p) @ gamma.
    """
    p = np.asarray(p, dtype=float)
    dp_dt = np.asarray(dp_dt, dtype=float)

    if p.ndim != 1 or dp_dt.shape != p.shape:
        raise ValueError("p and dp_dt must be one-dimensional arrays of equal size.")

    if np.any(p < 0):
        raise ValueError("Probabilities must be nonnegative.")

    A, edges = construct_transport_matrix(p, allowed_edges)

    result = lsq_linear(
        A,
        dp_dt,
        bounds=(0.0, np.inf),
        method="trf",
        tol=1e-12
    )

    Gamma = vector_to_rate_matrix(result.x, edges, len(p))

    return Gamma, result, A, edges

In [6]:
A,edges=construct_transport_matrix(np.array([0.5, 0.5]), allowed_edges=[(0, 1), (1, 0)])
Gamma, result, A, edges= solve_nonnegative_rates(np.array([0.5, 0.5]), np.array([-0.1, 0.1]), allowed_edges=[(0, 1), (1, 0)])
print(Gamma)

[[0.  0.1]
 [0.3 0. ]]


In [7]:
print(A)
#Funnily enough this is the correct solution with minimum norm that is coming out at the end.
print(np.linalg.pinv(A)@ np.array([-0.1, 0.1  ]))

[[ 0.5 -0.5]
 [-0.5  0.5]]
[-0.1  0.1]


We define two functions below:
1. NNLS using Moore-Penrose. This minimises the norm by using linear Algebra, but it does it step-wise rather than in a global sense.
2. SLSQP is guaranteed to give the globally optimal solution for what we want where we satisfy constraint, with non-negative rates and the best possible norm we can get.

Ultimately, we want to find out if these always give the same answer or not.

In [37]:
def nnls_pinv(A, b, tol=None, maxiter=None, rcond=1e-12):
    """
    Solve min ||A @ x - b||_2 subject to x >= 0 using an
    active-set algorithm.

    Each passive-set least-squares problem is solved explicitly with
    the Moore-Penrose pseudoinverse:

        x_P = pinv(A[:, P]) @ b

    Returns
    -------
    x : ndarray
        Nonnegative NNLS solution.

    residual_norm : float
        ||A @ x - b||_2.

    info : dict
        Diagnostic information.
    """
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    if A.ndim != 2:
        raise ValueError("A must be a two-dimensional array.")

    m, n = A.shape

    if b.shape != (m,):
        raise ValueError(f"b must have shape ({m},).")

    if tol is None:
        tol = (
            10
            * np.finfo(float).eps
            * max(m, n)
            * max(np.linalg.norm(A, 1), 1.0)
            * max(np.linalg.norm(b), 1.0)
        )

    if maxiter is None:
        maxiter = 30 * n

    # P[j] = True: variable j is free.
    # P[j] = False: variable j is fixed at zero.
    P = np.zeros(n, dtype=bool)

    x = np.zeros(n)
    iterations = 0

    while True:
        residual = b - A @ x

        # Negative gradient of 1/2 ||Ax-b||^2
        w = A.T @ residual

        # Variables fixed to zero that violate the KKT condition.
        candidates = np.flatnonzero((~P) & (w > tol))

        if len(candidates) == 0:
            break

        # Move the variable with the largest KKT violation into P.
        entering = candidates[np.argmax(w[candidates])]
        P[entering] = True

        iterations += 1

        if iterations > maxiter:
            raise RuntimeError("Maximum number of iterations reached.")

        while True:
            z = np.zeros(n)

            if np.any(P):
                A_P = A[:, P]

                # Moore-Penrose solution on the current free set.
                z[P] = np.linalg.pinv(
                    A_P,
                    rcond=rcond
                ) @ b

            # If all free components are positive, accept the step.
            if np.all(z[P] > tol):
                x = z
                break

            # Otherwise move toward z until one component reaches zero.
            bad = P & (z <= tol)
            denominator = x - z
            valid = bad & (denominator > tol)

            if not np.any(valid):
                # Degenerate zero components: return them to the
                # zero-constrained set.
                P[bad] = False
                x[bad] = 0.0

                if not np.any(P):
                    x[:] = 0.0
                    break

                continue

            alpha = np.min(
                x[valid] / denominator[valid]
            )

            x = x + alpha * (z - x)

            # Components that have reached zero leave the free set.
            leaving = P & (x <= tol)
            x[leaving] = 0.0
            P[leaving] = False

            iterations += 1

            if iterations > maxiter:
                raise RuntimeError("Maximum number of iterations reached.")

    # Remove tiny numerical negative values.
    x[np.abs(x) <= tol] = 0.0

    residual = b - A @ x
    w = A.T @ residual

    info = {
        "iterations": iterations,
        "positive_set": np.flatnonzero(x > tol),
        "zero_set": np.flatnonzero(x <= tol),
        "dual_vector": w,
        "kkt_satisfied": (
            np.all(x >= -tol)
            and np.all(w[x <= tol] <= tol)
            and np.all(np.abs(w[x > tol]) <= 10 * tol)
        )
    }

    return x, np.linalg.norm(residual), info

def minimum_norm_positive_solution(
    A,
    b,
    *,
    epsilon=0.0,
    ftol=1e-12,
    maxiter=2000,
):
    """
    Solve

        minimise    0.5 * ||x||_2^2
        subject to  A @ x = b
                    x >= epsilon

    Parameters
    ----------
    A : array_like, shape (m, n)
        Constraint matrix.

    b : array_like, shape (m,)
        Target vector.

    epsilon : float
        Lower bound on every component of x.
        Use:
            epsilon = 0.0   for x >= 0
            epsilon = 1e-10 for approximately x > 0

    ftol : float
        SLSQP stopping tolerance.

    maxiter : int
        Maximum number of SLSQP iterations.

    Returns
    -------
    result : scipy.optimize.OptimizeResult
        The optimal vector is result.x.
    """

    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)

    if A.ndim != 2:
        raise ValueError("A must be a two-dimensional matrix.")

    m, n = A.shape

    if b.shape != (m,):
        raise ValueError(f"b must have shape ({m},), not {b.shape}.")

    if epsilon < 0:
        raise ValueError("epsilon must be non-negative.")

    lower = np.full(n, epsilon)
    upper = np.full(n, np.inf)

    # ---------------------------------------------------------------
    # First find a feasible starting point satisfying:
    #
    #     A @ x = b
    #     x >= epsilon
    #
    # linprog is used only as a feasibility solver here.
    # ---------------------------------------------------------------
    feasible = linprog(
        c=np.zeros(n),
        A_eq=A,
        b_eq=b,
        bounds=list(zip(lower, upper)),
        method="highs",
    )

    if not feasible.success:
        raise ValueError(
            "No vector satisfying both A @ x = b and "
            f"x >= {epsilon} was found.\n"
            f"Feasibility solver message: {feasible.message}"
        )

    
    x0 = feasible.x

    # Objective: 0.5 * ||x||^2
    def objective(x):
        return 0.5 * np.dot(x, x)

    # Exact analytical gradient of the objective.
    def objective_gradient(x):
        return x

    equality_constraint = LinearConstraint(
        A,
        lb=b,
        ub=b,
    )

    result = minimize(
        fun=objective,
        x0=x0,
        jac=objective_gradient,
        method="SLSQP",
        bounds=Bounds(lower, upper),
        constraints=[equality_constraint],
        options={
            "ftol": ftol,
            "maxiter": maxiter,
            "disp": False,
        },
    )


    if not result.success:
        raise RuntimeError(
            f"SLSQP failed with status {result.status}: {result.message}"
        )
    # Explicit diagnostics
    x = result.x
    residual = A @ x - b

    result.residual = residual
    result.residual_norm = np.linalg.norm(residual)
    result.solution_norm = np.linalg.norm(x)
    result.minimum_component = np.min(x)

    return result



def full_result_compare(A, b):
    #Method 1
    v, residual, info = nnls_pinv(A, b)
    #Enforcing postivity vs. not happens to give the same rates here.
    print("Small norm and positive rates:")
    print("v =", v)
    print("A v =", A @ v)
    print("Residual =", residual)
    print("Positive set =", info["positive_set"])
    print("KKT satisfied =", info["kkt_satisfied"])

    #Method 2
    print(" ")
    print("Results when we ask for min norm w/o positivity:")
    result = lsq_linear(A, b)
    print("v =", result.x)
    print("A v =", A @ result.x)
    print("Residual =", A @ result.x-b)

    #Method 3
    print(" ")
    print("Most optimal solution with positivity and global minimum norm:")
    result = minimum_norm_positive_solution(A,b)
    print("v =", result.x)
    print("A v =", A @ result.x)
    print("Residual =", A @ result.x-b)

def MPNNLSvsSLSQP(A,b):
    #Method 1
    v, residual, info = nnls_pinv(A, b)
    result = minimum_norm_positive_solution(A,b)

    print("Difference between MPNNLS and SLSQP solutions:")
    print("v (MPNNLS) =", v)
    print("v (SLSQP) =", result.x)
    print("Difference =", np.sum(np.abs(v - result.x)**2))


### 1. Qubit example (tug of war)
The particular values we've chosen below are completely generic for this case. The min norm solution and the positive + min norm solution do not agree with each other as expected.

The minimum norm part does further constrain the positivity restricted solution, since otherwise it could be anywhere on the semi-infinite line defined by tuning the total rate.

In [33]:
A, edges = construct_transport_matrix(np.array([0.5, 0.5]), allowed_edges=[(0, 1), (1, 0)])

b = np.array([-0.1, 0.1])

full_result_compare(A,b)

Small norm and positive rates:
v = [0.  0.2]
A v = [-0.1  0.1]
Residual = 3.925231146709438e-17
Positive set = [1]
KKT satisfied = True
 
Results when we ask for min norm w/o positivity:
v = [-0.1  0.1]
A v = [-0.1  0.1]
Residual = [ 5.55111512e-17 -5.55111512e-17]
 
Most optimal solution with positivity and global minimum norm:
v = [0.  0.2]
A v = [-0.1  0.1]
Residual = [0. 0.]


### 2. Qutrit example (cycle)
The Qutrit example is more complicated, since now we have a single loop gauge that we can adjust.

In [38]:
A, edges = construct_transport_matrix(np.array([1/3, 1/3, 1/3]))


b = np.array([-1/3, 0.0, 1/3])

full_result_compare(A,b)
MPNNLSvsSLSQP(A,b)
result = minimum_norm_positive_solution(A, b)

print("success:", result.success)
print("status:", result.status)
print("message:", result.message)
print("iterations:", result.nit)

Small norm and positive rates:
v = [0. 0. 0. 0. 1. 0.]
A v = [-0.33333333  0.          0.33333333]
Residual = 7.850462293418876e-17
Positive set = [4]
KKT satisfied = True
 
Results when we ask for min norm w/o positivity:
v = [-0.16666667 -0.33333333  0.16666667 -0.16666667  0.33333333  0.16666667]
A v = [-3.33333333e-01  1.23358114e-17  3.33333333e-01]
Residual = [-5.55111512e-17  1.23358114e-17  5.55111512e-17]
 
Most optimal solution with positivity and global minimum norm:


RuntimeError: SLSQP failed with status 6: Singular matrix C in LSQ subproblem

In [62]:
A, edges = construct_transport_matrix(np.array([1/3, 1/3, 1/3]))
b = np.array([-1/3, 0.0, 1/3])

#The issue is that this matrix is rank-deficient
print(A)


#Using the Moore-penrose inverse,
v1=np.linalg.pinv(A)@ np.array([-1/3, 0.0, 1/3])
print("v1=",v1)
print(vector_to_rate_matrix(v1, edges, 3))

#Find active set and remove these columns from A and edges to get a full-rank matrix
activeset1=np.transpose(np.argwhere(v1>0))[0]
A1=A[:,activeset1]
print(activeset1)

#Now rinse and repeat
v2=np.linalg.pinv(A1)@ np.array([-1/3, 0.0, 1/3])
print(v2)




[[ 0.33333333  0.33333333 -0.33333333  0.         -0.33333333  0.        ]
 [-0.33333333  0.          0.33333333  0.33333333  0.         -0.33333333]
 [ 0.         -0.33333333  0.         -0.33333333  0.33333333  0.33333333]]
v1= [-0.16666667 -0.33333333  0.16666667 -0.16666667  0.33333333  0.16666667]
[[ 0.         -0.16666667 -0.33333333]
 [ 0.16666667  0.         -0.16666667]
 [ 0.33333333  0.16666667  0.        ]]
[2 4 5]
[0.33333333 0.66666667 0.33333333]


The issue is that with the sequential Moore-Penrose + non-negative constraint finding the active set, we are not guaranteed to get the globally best solution. To guarantee it, we can use a Sequential Least Squares Quadratic Programming (SLSQP). 